# 16a — MP-Declare Constraint Mining (RuM) — Helpdesk

Mines MP-Declare constraints with data conditions from the Helpdesk event log using RuM's MINERful + MpEnhancer.
Categorizes into prefix-safe vs sequence constraints and saves to pkl for downstream notebooks.

In [1]:
import sys
import os
from pathlib import Path

os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home'

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))

# src/ must be on sys.path for torch.load to unpickle event_log_loader classes
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from src.interpretability.perturbation_methods import csv_to_xes, discover_mpdeclare
from src.interpretability.perturbation_methods import mpdeclare_to_declare_constraints
from src.interpretability.perturbation_methods import DeclareConstraintChecker

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:163: DeprecationWarning: module 'sre_parse' is deprecated
  import sre_parse
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/lark/utils.py:164: DeprecationWarning: module 'sre_constants' is deprecated
  import sre_constants


In [2]:
# ===== CONFIGURATION =====

# Declare mining hyperparameters
MIN_SUPPORT = 0.95
DATA_CONDITIONS = 'ACTIVATIONS'

In [3]:
csv_path = _current / 'data' / 'helpdesk.csv'
xes_path = _current / 'data' / 'helpdesk.xes'

if not xes_path.exists():
    csv_to_xes(csv_path, xes_path, case_id_col='Case ID', activity_col='Activity', timestamp_col='Complete Timestamp')
    print(f'Converted CSV to XES: {xes_path}')
else:
    print(f'XES file already exists: {xes_path}')

XES file already exists: /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/data/helpdesk.xes


In [4]:
# Start JVM with extra heap before discover_mpdeclare
import jpype
if not jpype.isJVMStarted():
    from src.interpretability.perturbation_methods.revised_plus.rum_mpdeclare import RUM_JAR
    jpype.startJVM(f'-Djava.class.path={RUM_JAR}', '-Djava.awt.headless=true', '-Xmx4g', convertStrings=True)
    __import__('jpype.imports')

mpdeclare_constraints = discover_mpdeclare(
    xes_path,
    min_support=MIN_SUPPORT,
    data_conditions=DATA_CONDITIONS,
)
print(f'Mined {len(mpdeclare_constraints)} MP-Declare constraints')

MP-Declare discovery:   0%|          | 0/3 [00:00<?, ?it/s]

log4j:WARN No appenders could be found for logger (minerful.miner.core.MinerFulKBCore).
log4j:WARN Please initialize the log4j system properly.
log4j:WARN See http://logging.apache.org/log4j/1.2/faq.html#noconfig for more info.


||||||||||||||||||||||||||||||||||||||||


Converting to RuM format:   0%|          | 0/245 [00:00<?, ?it/s]

2026-03-08 18:51:40,208 INFO    [main] task.discovery.mp_enhancer.MpEnhancer - MpEnhancer (588449070) started at: 1772992300206
2026-03-08 18:51:40,212 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Number of constraints to process: 14
2026-03-08 18:51:40,214 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 1: Constraint(supp=0.9963548): Response[Assign seriousness, Closed] | |


Mar 08, 2026 6:51:42 PM com.github.fommil.jni.JniNamer arch
Mar 08, 2026 6:51:42 PM com.github.fommil.netlib.ARPACK <clinit>
Mar 08, 2026 6:51:42 PM com.github.fommil.jni.JniNamer arch
Mar 08, 2026 6:51:42 PM com.github.fommil.netlib.ARPACK <clinit>


2026-03-08 18:51:53,902 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 2: Constraint(supp=0.9822912): Precedence[Closed, Assign seriousness] | |
2026-03-08 18:51:53,927 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 3: Constraint(supp=0.98507464): Precedence[Create SW anomaly, Assign seriousness] | |
2026-03-08 18:51:53,950 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 4: Constraint(supp=1.0): Precedence[Require upgrade, Assign seriousness] | |
2026-03-08 18:51:53,968 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 5: Constraint(supp=0.9983799): Response[Assign seriousness, Resolve ticket] | |
2026-03-08 18:52:05,893 DEBUG   [main] task.discovery.mp_enhancer.CorrelationMiner - Processing constraint 6: Constraint(supp=0.98133653): Precedence[Resolve ticket, Assign seriousness] | |
2026-03-08 18:52:05,962 DEBUG   [main] task.discovery.mp_enhancer

Extracting results:   0%|          | 0/23 [00:00<?, ?it/s]

Mined 23 MP-Declare constraints


In [5]:
# Build activity vocabulary from dataset
import torch
from tqdm.auto import tqdm
from src.interpretability.utils.tensor_decoder import TensorDecoder

with tqdm(total=2, desc='Loading dataset') as pbar:
    pbar.set_postfix_str('loading pkl...')
    data_path = _current / 'encoded_data' / 'test_philipp' / 'helpdesk_all_5_test.pkl'
    full_dataset = torch.load(data_path, weights_only=False)
    pbar.update(1)

    pbar.set_postfix_str('building vocabulary...')
    decoder = TensorDecoder(full_dataset)

    ACTIVITY_FEATURE = 'Activity'
    activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
    max_idx = max(activity_idx_to_label.keys())
    activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]

    # Build name -> index mapping for conversion
    activity_name_to_idx = {name: i for i, name in enumerate(activity_names)}
    pbar.update(1)
    pbar.set_postfix_str('done')

print(f'Activity vocabulary ({len(activity_names)}):')
for i, name in enumerate(activity_names):
    print(f'  {i}: {name}')

Loading dataset:   0%|          | 0/2 [00:00<?, ?it/s]

Activity vocabulary (16):
  0: <pad>
  1: Assign seriousness
  2: Closed
  3: Create SW anomaly
  4: DUPLICATE
  5: EOS
  6: INVALID
  7: Insert ticket
  8: RESOLVED
  9: Require upgrade
  10: Resolve SW anomaly
  11: Resolve ticket
  12: Schedule intervention
  13: Take in charge ticket
  14: VERIFIED
  15: Wait


/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator OrdinalEncoder from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.7.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/philippeichhorn/.local/share/v

In [6]:
# Convert MPDeclareConstraint -> DeclareConstraint
from tqdm.auto import tqdm

with tqdm(total=3, desc='Post-processing constraints') as pbar:
    pbar.set_postfix_str('converting to DeclareConstraint...')
    all_constraints, data_conditions = mpdeclare_to_declare_constraints(
        mpdeclare_constraints, activity_name_to_idx
    )
    pbar.update(1)

    pbar.set_postfix_str('categorizing...')
    prefix_safe_constraints = set(DeclareConstraintChecker.prefix_safe_constraints(all_constraints))
    sequence_constraints = all_constraints - prefix_safe_constraints
    pbar.update(1)

    pbar.set_postfix_str('done')
    pbar.update(1)

print(f'Total constraints:        {len(all_constraints)}')
print(f'Prefix-safe constraints:  {len(prefix_safe_constraints)}')
print(f'Sequence constraints:     {len(sequence_constraints)}')
print(f'Data conditions:          {len(data_conditions)}')

Post-processing constraints:   0%|          | 0/3 [00:00<?, ?it/s]

Total constraints:        23
Prefix-safe constraints:  16
Sequence constraints:     7
Data conditions:          11


In [7]:
# Display all three categories
print('=' * 80)
print('PREFIX-SAFE CONSTRAINTS (monotonic violations — reliable for prefix scoring)')
print('=' * 80)
for c in sorted(prefix_safe_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f'  {c.format(activity_names)}')

print(f"\n{'=' * 80}")
print('SEQUENCE CONSTRAINTS (require full trace — used as validity gate)')
print('=' * 80)
for c in sorted(sequence_constraints, key=lambda x: (x.template.value, x.activities)):
    print(f'  {c.format(activity_names)}')

print(f"\n{'=' * 80}")
print('DATA CONDITIONS')
print('=' * 80)
if data_conditions:
    for dc, cond in sorted(data_conditions.items(), key=lambda x: (x[0].template.value, x[0].activities)):
        print(f'  {dc.format(activity_names)}')
        print(f'    Condition: {cond}')
else:
    print('  (none)')

PREFIX-SAFE CONSTRAINTS (monotonic violations — reliable for prefix scoring)
  absence(Create SW anomaly, n=1)
  absence(DUPLICATE, n=1)
  absence(INVALID, n=1)
  absence(Insert ticket, n=1)
  absence(RESOLVED, n=1)
  absence(Require upgrade, n=1)
  absence(Resolve SW anomaly, n=1)
  absence(Schedule intervention, n=1)
  absence(VERIFIED, n=1)
  precedence(Assign seriousness, Closed)
  precedence(Assign seriousness, Create SW anomaly)
  precedence(Assign seriousness, Require upgrade)
  precedence(Assign seriousness, Resolve ticket)
  precedence(Assign seriousness, Take in charge ticket)
  precedence(Assign seriousness, Wait)
  precedence(Resolve ticket, Closed)

SEQUENCE CONSTRAINTS (require full trace — used as validity gate)
  response(Assign seriousness, Closed)
  response(Assign seriousness, Resolve ticket)
  response(Create SW anomaly, Resolve ticket)
  response(Require upgrade, Resolve ticket)
  response(Resolve ticket, Closed)
  response(Wait, Closed)
  response(Wait, Resolve ti

In [8]:
# Save to pkl
import pickle

constraints_pkl = {
    'all': all_constraints,
    'prefix_safe': prefix_safe_constraints,
    'sequence': sequence_constraints,
    'data_conditions': data_conditions,
    'activity_names': activity_names,
}

pkl_path = _current / 'encoded_data' / 'helpdesk_constraints.pkl'

with open(pkl_path, 'wb') as f:
    pickle.dump(constraints_pkl, f)

print(f'Saved constraints to {pkl_path}')
print(f'  all: {len(constraints_pkl["all"])} constraints')
print(f'  prefix_safe: {len(constraints_pkl["prefix_safe"])} constraints')
print(f'  sequence: {len(constraints_pkl["sequence"])} constraints')
print(f'  data_conditions: {len(constraints_pkl["data_conditions"])} entries')
print(f'  activity_names: {len(constraints_pkl["activity_names"])} activities')

Saved constraints to /Users/philippeichhorn/IdeaProjects/XAI-Probabilistic_Suffix_Prediction_U-ED-LSTM_pub/encoded_data/helpdesk_constraints.pkl
  all: 23 constraints
  prefix_safe: 16 constraints
  sequence: 7 constraints
  data_conditions: 11 entries
  activity_names: 16 activities
